# 02 — Explore the yoked dataset

Pick a rat, pick one of its acquisition sessions, and watch the recorded behaviour replay through `CornerMazeEnv`.

## How to use

1. **Group** — training cohort (`PI`, `PI+VC`, `PI+VC_f1`, `VC`).
2. **Session type** — the upstream label (`Fixed Cue 1`, `Dark Train`, etc.). The cell below maps each (group, session type) pair to the matching env paradigm.
3. **Sessions** — scrollable list of every session in the dataset for that combo, one row per `(subject, session_number)`. Highlight one.
4. **Run** — builds the env for that session and arms playback. **Play** then steps the rat's recorded actions through the env at the speed set by the slider.

Only `session_phase = Acquisition` sessions are listed — Exposure has no env paradigm mapping yet (see [src/corner_maze_rl/data/session_types.py](../src/corner_maze_rl/data/session_types.py)).

In [1]:
# Setup: assumes `pip install -e .` has been run from the repo root.
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q --upgrade --force-reinstall --no-deps git+https://github.com/ryangrg/corner-maze-rl.git
    !pip install -q minigrid duckdb

import io
import json
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
from PIL import Image as PILImage

from corner_maze_rl.data.load import (
    YokedPaths,
    load_actions_for_session,
    load_sessions,
    load_subjects,
)
from corner_maze_rl.data.session_types import (
    PARADIGM_MAP,
    map_session_to_env_kwargs,
)
from corner_maze_rl.env.constants import EYE_IMG_SIZE
from corner_maze_rl.env.corner_maze_env import CornerMazeEnv


def _find_repo_root(start: Path | None = None) -> Path:
    """Walk up from ``start`` (default CWD) to the first dir containing
    ``pyproject.toml``. VS Code's Jupyter extension launches notebooks
    with CWD = the notebook's directory, so relative paths like
    ``data/yoked/dataset`` need to be anchored explicitly."""
    p = (start or Path.cwd()).resolve()
    for d in (p, *p.parents):
        if (d / "pyproject.toml").is_file():
            return d
    return p


REPO_ROOT = _find_repo_root()
DATASET_DIR = REPO_ROOT / "data" / "yoked" / "dataset"

# --- Load the yoked dataset once ---
PATHS = YokedPaths.from_dir(DATASET_DIR, actions_variant="synthetic_pretrial")
PATHS.assert_exist()
SUBJECTS = load_subjects(PATHS)
SESSIONS = load_sessions(PATHS)

# --- Acquisition sessions only, joined to subject metadata ---
ACQ = SESSIONS[SESSIONS["session_phase"] == "Acquisition"].merge(
    SUBJECTS[["subject_id", "subject_name", "training_group", "cue_goal_orientation"]],
    on="subject_id",
    suffixes=("", "_subj"),
)
GROUPS = sorted(ACQ["training_group"].unique().tolist())

# Sanity: every (group, session_type) pair below must be in PARADIGM_MAP.
_present = ACQ.groupby(["training_group", "session_type"]).size().reset_index(name="n")
_unmapped = [
    (g, st) for g, st in zip(_present["training_group"], _present["session_type"])
    if (g, st) not in PARADIGM_MAP
]
if _unmapped:
    print(f"Warning: {len(_unmapped)} (group, session_type) pair(s) in the dataset are not in PARADIGM_MAP "
          f"and will be skipped if selected: {_unmapped}")

print(f"Repo root:     {REPO_ROOT}")
print(f"Dataset dir:   {DATASET_DIR}")
print(f"Acquisition sessions: {len(ACQ)} across {len(GROUPS)} groups.")
for g in GROUPS:
    sub = ACQ[ACQ["training_group"] == g]
    types = sub["session_type"].value_counts().to_dict()
    print(f"  {g:<10} {len(sub):>3} sessions — {types}")

/Users/ryangrgurich/venvs/ai-venv/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Repo root:     /Users/ryangrgurich/Code/python-dev/corner-maze-rl
Dataset dir:   /Users/ryangrgurich/Code/python-dev/corner-maze-rl/data/yoked/dataset
Acquisition sessions: 422 across 4 groups.
  PI         121 sessions — {'Dark Train': 117, 'Fixed Cue 1': 4}
  PI+VC      118 sessions — {'Fixed Cue 1': 118}
  PI+VC_f1    68 sessions — {'Fixed Cue 1 Twist': 68}
  VC         115 sessions — {'Rotate Train': 115}


## Pick a session and replay

The cell below is the interactive viewer. Each tick of **Play** consumes one action from the rat's recorded sequence, calls `env.step(action)`, and re-renders the maze + eye images. The trial counter advances on each rewarded step (per the dataset's `rewarded` column).

In [3]:
#@title Yoked-session replay viewer

import asyncio

# --- Stage the eye-image parquet under REPO_ROOT so the env's
# _load_embeddings() finds it regardless of CWD (it searches
# `repo_root_via___file__ / EMBEDDING_PARQUET_PATH`). ---
EYE_PARQUET = REPO_ROOT / "data/dataframes/dual-indep-20260319-222411-embeddings-allposes.parquet"
EYE_URL = ("https://github.com/ryangrg/corner-maze-rl-legacy/raw/main/"
           "data/dataframes/dual-indep-20260319-222411-embeddings-allposes.parquet")
if not EYE_PARQUET.is_file():
    EYE_PARQUET.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading eye-image embeddings (~2 MB) → {EYE_PARQUET}")
    urllib.request.urlretrieve(EYE_URL, EYE_PARQUET)

# --- Theme + constants ---
BG_COLOR = "#333333"
TEXT_COLOR = "#ffffff"
MUTED_COLOR = "#aaaaaa"
DIR_NAMES = {0: "East", 1: "South", 2: "West", 3: "North"}
ACTION_NAMES = {0: "Left", 1: "Right", 2: "Forward", 3: "Enter Well", 4: "Pause"}
GOAL_IDX_TO_DIR = {0: "NE", 1: "SE", 2: "SW", 3: "NW"}
IMG_SIZE = 299  # 13 * 23
FIELD_W = "320px"
DEFAULT_GROUP = "PI+VC" if "PI+VC" in GROUPS else GROUPS[0]

style_widget = widgets.HTML(f"""<style>
.maze-ui, .maze-ui * {{
    background-color: {BG_COLOR} !important;
    color: {TEXT_COLOR} !important;
    border-color: {BG_COLOR} !important;
}}
.maze-ui {{
    padding: 15px !important;
    border-radius: 0 !important;
    margin: 0 !important;
    width: 100% !important;
    box-sizing: border-box !important;
}}
.maze-ui .widget-image {{ background-color: transparent !important; }}
.maze-ui .section-divider {{
    height: 1px !important;
    background-color: #888 !important;
    margin: 12px 0 !important;
    width: 100% !important;
}}
.maze-ui .form-input select,
.maze-ui .form-input input {{
    background-color: #4a4a4a !important;
    color: #ffffff !important;
    border: 1px solid #888 !important;
    border-color: #888 !important;
    border-radius: 4px !important;
    padding: 2px 6px !important;
}}
.maze-ui .form-button,
.maze-ui button.form-button,
.maze-ui .form-button > button,
.maze-ui .form-button > .jupyter-button {{
    background: #6a6a6a !important;
    background-image: none !important;
    background-color: #6a6a6a !important;
    color: #ffffff !important;
    border: 1px solid #d0d0d0 !important;
    border-color: #d0d0d0 !important;
    border-radius: 4px !important;
    box-shadow: inset 0 0 0 1px #d0d0d0 !important;
    font-weight: 500 !important;
    padding: 4px 12px !important;
}}
.maze-ui .form-button:hover {{
    background: #7a7a7a !important;
    background-color: #7a7a7a !important;
}}
.maze-ui .form-button * {{
    background: transparent !important;
    background-color: transparent !important;
    color: #ffffff !important;
}}
.maze-ui .form-field-label {{
    font-size: 12px !important;
    font-weight: 600 !important;
    color: #c8c8c8 !important;
    margin: 6px 0 2px 0 !important;
}}
.maze-ui .form-status {{
    font-size: 12px !important;
    color: #c8c8c8 !important;
    margin: 8px 0 0 0 !important;
    line-height: 1.5 !important;
}}
.maze-ui select option {{ background-color: #4a4a4a !important; color: #ffffff !important; }}
.cell-output-ipywidget-background:has(.maze-ui),
.jp-OutputArea-output:has(.maze-ui),
.output_subarea:has(.maze-ui),
.cell-output:has(.maze-ui) {{
    background: {BG_COLOR} !important;
    background-color: {BG_COLOR} !important;
    padding: 0 !important;
}}
</style>""")

# --- State held across callbacks ---
state = {
    "env": None,
    "actions": None,        # pd.DataFrame
    "trial_configs": [],
    "n_steps": 0,
    "current_step": 0,
    "trial_idx": 0,
    "cum_reward": 0.0,
    "seed": 42,
    "session_label_str": "(no session loaded)",
    "has_eyes": False,
}
play_state = {"running": False}

zero_eye = np.zeros((EYE_IMG_SIZE, EYE_IMG_SIZE), dtype=np.float64)
blank_maze = np.full((23 * 13, 23 * 13, 3), 30, dtype=np.uint8)


def frame_to_png(frame):
    img = PILImage.fromarray(frame).resize((IMG_SIZE, IMG_SIZE), PILImage.NEAREST)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()


def eye_to_png(eye_arr):
    gray = (np.asarray(eye_arr, dtype=np.float64) * 255).astype(np.uint8)
    img = PILImage.fromarray(gray).resize((IMG_SIZE, IMG_SIZE), PILImage.NEAREST)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()


# --- Catalog helpers ---
def list_session_types(group):
    sub = ACQ[(ACQ["training_group"] == group)]
    types = sorted(sub["session_type"].unique().tolist())
    # Keep only mapped pairs so the user can't pick a dead option.
    return [t for t in types if (group, t) in PARADIGM_MAP]


def list_session_options(group, session_type):
    sub = ACQ[(ACQ["training_group"] == group) & (ACQ["session_type"] == session_type)]
    sub = sub.sort_values(["subject_name", "session_number"])
    out = []
    for row in sub.itertuples():
        label = (
            f"{row.subject_name}  ·  session {int(row.session_number):>3}  "
            f"·  {int(row.n_trials):>2} trials"
        )
        out.append((label, int(row.session_id)))
    return out


# --- Display widgets ---
maze_widget = widgets.Image(format="png", width=IMG_SIZE, height=IMG_SIZE,
                            value=frame_to_png(blank_maze))
left_eye_widget = widgets.Image(format="png", width=IMG_SIZE, height=IMG_SIZE,
                                value=eye_to_png(zero_eye))
right_eye_widget = widgets.Image(format="png", width=IMG_SIZE, height=IMG_SIZE,
                                 value=eye_to_png(zero_eye))
session_name_label = widgets.HTML(value="")
pose_caption = widgets.HTML(value="")
info_label = widgets.HTML(value="")
trial_label = widgets.HTML(value="")
status_label = widgets.HTML(value="")
status_label.add_class("form-status")


def _muted(text):
    """Render a placeholder value in muted colour for the empty state."""
    return f'<span style="color:{MUTED_COLOR};">{text}</span>'


def render_state():
    """Always populate every panel — when no session is loaded the maze
    and eye panels go blank but the info / trial blocks still show their
    full skeleton with `—` placeholders, so the layout doesn't shift when
    a session is finally chosen."""
    env = state["env"]
    has_session = env is not None

    # --- Image panels ---
    if has_session:
        maze_widget.value = frame_to_png(env.get_allocentric_frame(tile_size=23))
        if state["has_eyes"]:
            try:
                pose = env._get_pose_label()
                le = env._pose_to_left_eye.get(pose, zero_eye)
                re = env._pose_to_right_eye.get(pose, zero_eye)
            except Exception:
                pose = "(no eyes)"
                le = re = zero_eye
        else:
            pose = "(eye images not loaded)"
            le = re = zero_eye
        left_eye_widget.value = eye_to_png(le)
        right_eye_widget.value = eye_to_png(re)
    else:
        maze_widget.value = frame_to_png(blank_maze)
        left_eye_widget.value = eye_to_png(zero_eye)
        right_eye_widget.value = eye_to_png(zero_eye)
        pose = "—"

    # --- Captions under the image strip ---
    session_name_label.value = (
        f'<div style="font-size:13px; text-align:center; margin-top:4px;">'
        f'<b>{state["session_label_str"]}</b></div>'
    )
    pose_caption.value = (
        f'<div style="font-size:13px; text-align:center; margin-top:4px;">'
        f'<code>{pose}</code></div>'
    )

    # --- Info block (always rendered, shows placeholders when empty) ---
    if has_session:
        x, y = env.agent_pos
        d = env.agent_dir
        step_html = f'{state["current_step"]} / {state["n_steps"]}'
        reward_html = f'{state["cum_reward"]:.4f}'

        # "Last action" = the action that brought the agent to the
        # currently-displayed state. At step 0 nothing has executed yet.
        # At step N>0 the action consumed was actions[N-1].
        last_idx = state["current_step"] - 1
        if last_idx >= 0 and state["actions"] is not None:
            la = int(state["actions"].iloc[last_idx]["action"])
            last_action_html = f'{la} ({ACTION_NAMES.get(la, "?")})'
        else:
            last_action_html = _muted("—")

        # Rat's recorded pre-action state at the current step (rows store
        # state BEFORE the action in that row is applied; our env is
        # in-sync when its post-step state matches actions[step].grid_x/y).
        # When current_step == n_steps the session is done and there is
        # no row to compare against.
        cs = state["current_step"]
        if state["actions"] is not None and cs < state["n_steps"]:
            ra = state["actions"].iloc[cs]
            rat_x = int(ra["grid_x"])
            rat_y = int(ra["grid_y"])
            rat_d = int(ra["direction"])
            rat_pos_html = f'({rat_x}, {rat_y}) {rat_d} ({DIR_NAMES.get(rat_d, "?")})'
            diverged = (int(x), int(y), int(d)) != (rat_x, rat_y, rat_d)
        else:
            rat_pos_html = _muted("—")
            diverged = False

        # Inline `!important` beats the blanket `.maze-ui *` rule.
        if diverged:
            pos_html = f'<span style="color:#ff5555 !important;">({x}, {y})</span>'
            dir_html = f'<span style="color:#ff5555 !important;">{d} ({DIR_NAMES.get(d, "?")})</span>'
        else:
            pos_html = f"({x}, {y})"
            dir_html = f'{d} ({DIR_NAMES.get(d, "?")})'
    else:
        pos_html = _muted("—")
        dir_html = _muted("—")
        step_html = _muted("0 / 0")
        reward_html = _muted("—")
        last_action_html = _muted("—")
        rat_pos_html = _muted("—")

    info_label.value = (
        f'<div style="font-size:14px; line-height:1.8;">'
        f'<b>Position:</b> {pos_html}<br>'
        f'<b>Direction:</b> {dir_html}<br>'
        f'<b>Rat pos:</b> {rat_pos_html}<br>'
        f'<b>Step:</b> {step_html}<br>'
        f'<b>Last action:</b> {last_action_html}<br>'
        f'<b>Reward:</b> {reward_html}'
        f'</div>'
    )

    # --- Trial block (always rendered) ---
    n_trials = len(state["trial_configs"])
    if not has_session or n_trials == 0:
        trial_html = _muted("—")
        goal_html = _muted("—")
    else:
        ti = state["trial_idx"]
        if ti < n_trials:
            trial_html = f'{ti + 1} / {n_trials}'
            goal_idx = int(state["trial_configs"][ti][2])
            goal_html = GOAL_IDX_TO_DIR.get(goal_idx, "?")
        else:
            trial_html = f'all {n_trials} complete'
            goal_html = _muted("—")

    trial_label.value = (
        f'<div style="font-size:14px; line-height:1.8;">'
        f'<b>Trial:</b> {trial_html}<br>'
        f'<b>Goal corner:</b> {goal_html}'
        f'</div>'
    )


# --- Form widgets ---
group_dd = widgets.Dropdown(
    options=GROUPS, value=DEFAULT_GROUP,
    layout=widgets.Layout(width=FIELD_W),
)
group_dd.add_class("form-input")

_initial_types = list_session_types(DEFAULT_GROUP)
session_type_dd = widgets.Dropdown(
    options=_initial_types,
    value=_initial_types[0] if _initial_types else None,
    layout=widgets.Layout(width=FIELD_W),
)
session_type_dd.add_class("form-input")

_initial_options = list_session_options(DEFAULT_GROUP, _initial_types[0]) if _initial_types else []
sessions_select = widgets.Select(
    options=_initial_options,
    value=_initial_options[0][1] if _initial_options else None,
    rows=10,
    layout=widgets.Layout(width=FIELD_W),
)
sessions_select.add_class("form-input")


def _on_group_change(change):
    types = list_session_types(change["new"])
    session_type_dd.options = types
    if types:
        session_type_dd.value = types[0]


def _on_session_type_change(change):
    if change["new"] is None:
        sessions_select.options = []
        return
    opts = list_session_options(group_dd.value, change["new"])
    sessions_select.options = opts
    if opts:
        sessions_select.value = opts[0][1]


group_dd.observe(_on_group_change, names="value")
session_type_dd.observe(_on_session_type_change, names="value")

# --- Playback controls ---
speed_slider = widgets.IntSlider(
    value=20, min=1, max=120, step=1,
    description="Steps/sec",
    continuous_update=True,
    layout=widgets.Layout(width=FIELD_W),
)

progress = widgets.IntProgress(
    value=0, min=0, max=1,
    layout=widgets.Layout(width=FIELD_W),
)

run_button = widgets.Button(description="Run", layout=widgets.Layout(height="34px", width="120px"))
run_button.add_class("form-button")
reset_button = widgets.Button(description="Reset", icon="undo",
                              layout=widgets.Layout(height="34px", width="120px"))
reset_button.add_class("form-button")
play_button = widgets.Button(description="▶ Play",
                             layout=widgets.Layout(height="34px", width="120px"))
play_button.add_class("form-button")
back_button = widgets.Button(description="← Back",
                             layout=widgets.Layout(height="34px", width="90px"))
back_button.add_class("form-button")
forward_button = widgets.Button(description="Step →",
                                layout=widgets.Layout(height="34px", width="90px"))
forward_button.add_class("form-button")


def _set_status(html, color=MUTED_COLOR):
    status_label.value = f'<div style="color:{color};">{html}</div>'


def _stop_play():
    play_state["running"] = False
    play_button.description = "▶ Play"


def _advance_one_step() -> bool:
    """Consume one action from the recorded sequence; return False when the
    session is exhausted or the env terminates."""
    if state["env"] is None or state["actions"] is None:
        return False
    if state["current_step"] >= state["n_steps"]:
        return False
    row = state["actions"].iloc[state["current_step"]]
    try:
        _obs, reward, terminated, truncated, _info = state["env"].step(int(row.action))
    except Exception as exc:  # noqa: BLE001
        _set_status(f"env.step failed at step {state['current_step']}: {exc}", color="#f88")
        return False
    state["cum_reward"] += float(reward)
    if int(row.rewarded) == 1:
        state["trial_idx"] += 1
    state["current_step"] += 1
    if terminated or truncated:
        return False
    return True


def _replay_to(target_step: int) -> None:
    """Step backward by resetting the env and replaying forward to
    ``target_step``. CornerMazeEnv has no rewind primitive, so we eat the
    full O(target_step) cost — about 1.3 ms per step, so even rewinding
    1500 steps is under 2 s. Caller is responsible for stopping playback
    and re-rendering."""
    env = state["env"]
    if env is None or state["actions"] is None:
        return
    target_step = max(0, min(target_step, state["n_steps"]))
    env.reset(seed=state["seed"])
    state["current_step"] = 0
    state["trial_idx"] = 0
    state["cum_reward"] = 0.0
    while state["current_step"] < target_step:
        if not _advance_one_step():
            break


async def _play_loop():
    """Async tick driver. Reads speed_slider.value each iteration so speed
    changes apply mid-play."""
    while play_state["running"]:
        advanced = _advance_one_step()
        progress.value = state["current_step"]
        render_state()
        if not advanced:
            break
        sps = max(1, int(speed_slider.value))
        await asyncio.sleep(1.0 / sps)
    _stop_play()


def on_run_click(_b):
    _stop_play()
    sid = sessions_select.value
    if sid is None:
        _set_status("Pick a session first.", color="#f88")
        return

    sess_row = SESSIONS[SESSIONS["session_id"] == sid].iloc[0]
    subj_row = SUBJECTS[SUBJECTS["subject_id"] == int(sess_row["subject_id"])].iloc[0]

    raw_tc = sess_row["trial_configs"]
    try:
        trial_configs = json.loads(raw_tc) if isinstance(raw_tc, str) else list(raw_tc)
    except (TypeError, json.JSONDecodeError):
        trial_configs = []

    first_goal = (
        GOAL_IDX_TO_DIR.get(int(trial_configs[0][2])) if trial_configs else None
    )
    env_kwargs = map_session_to_env_kwargs(
        training_group=str(subj_row["training_group"]),
        yoked_session_type=str(sess_row["session_type"]),
        cue_goal_orientation=str(subj_row["cue_goal_orientation"]),
        start_goal_location=first_goal,
        # Pin the env's start arm + cue + goal sequence to the rat's
        # actual trials so trigger cells fire at the same step the rat
        # experienced them. Without this the env shuffles its own pool
        # by session_type and the agent's recorded trajectory desyncs
        # from the env's barriers/cues from step 0.
        trial_configs=trial_configs,
    )
    if env_kwargs is None:
        _set_status(
            f"No env mapping for ({subj_row['training_group']}, {sess_row['session_type']}).",
            color="#f88",
        )
        return

    try:
        env = CornerMazeEnv(render_mode="rgb_array", max_steps=10000, **env_kwargs)
        seed = int(sess_row.get("seed", 42))
        env.reset(seed=seed)
    except Exception as exc:  # noqa: BLE001
        _set_status(f"Could not build env: {type(exc).__name__}: {exc}", color="#f88")
        return

    has_eyes = False
    try:
        env._load_embeddings()
        has_eyes = True
    except Exception:
        pass

    actions_df = load_actions_for_session(PATHS, int(sid))

    state["env"] = env
    state["actions"] = actions_df
    state["trial_configs"] = trial_configs
    state["n_steps"] = len(actions_df)
    state["current_step"] = 0
    state["trial_idx"] = 0
    state["cum_reward"] = 0.0
    state["seed"] = seed
    state["has_eyes"] = has_eyes
    state["session_label_str"] = (
        f'{subj_row["subject_name"]} · session {int(sess_row["session_number"])} '
        f'· {sess_row["session_type"]} → {env_kwargs["session_type"]}'
    )

    progress.value = 0
    progress.max = max(1, state["n_steps"])

    _set_status(
        f"Loaded {state['n_steps']} actions across {len(trial_configs)} trials. Press Play."
    )
    render_state()


def on_play_click(_b):
    if state["env"] is None:
        _set_status("Run a session first.", color="#f88")
        return
    if play_state["running"]:
        _stop_play()
        return
    if state["current_step"] >= state["n_steps"]:
        # Auto-rewind so a second click of Play replays from the start.
        on_reset_click(None)
    play_state["running"] = True
    play_button.description = "⏸ Pause"
    asyncio.ensure_future(_play_loop())


def on_forward_click(_b):
    if state["env"] is None:
        _set_status("Run a session first.", color="#f88")
        return
    _stop_play()
    _advance_one_step()
    progress.value = state["current_step"]
    render_state()


def on_back_click(_b):
    if state["env"] is None:
        _set_status("Run a session first.", color="#f88")
        return
    _stop_play()
    if state["current_step"] == 0:
        return
    target = state["current_step"] - 1
    # For long sessions a back-step costs O(target_step) env.step calls
    # (~1.3 ms each); show a hint while it's running so the freeze is
    # explained.
    if target > 200:
        _set_status(f"Rewinding to step {target}…")
    _replay_to(target)
    progress.value = state["current_step"]
    _set_status(f"At step {state['current_step']}.")
    render_state()


def on_reset_click(_b):
    if state["env"] is None:
        return
    _stop_play()
    state["env"].reset(seed=state["seed"])
    state["current_step"] = 0
    state["trial_idx"] = 0
    state["cum_reward"] = 0.0
    progress.value = 0
    _set_status("Reset to step 0.")
    render_state()


run_button.on_click(on_run_click)
reset_button.on_click(on_reset_click)
play_button.on_click(on_play_click)
back_button.on_click(on_back_click)
forward_button.on_click(on_forward_click)

# --- Layout ---
col_style = widgets.Layout(width=f"{IMG_SIZE}px", align_items="center")
maze_col = widgets.VBox(
    [widgets.HTML("<b>Maze</b>"), maze_widget, session_name_label],
    layout=col_style,
)
left_eye_col = widgets.VBox(
    [widgets.HTML("<b>Left Eye</b>"), left_eye_widget], layout=col_style,
)
right_eye_col = widgets.VBox(
    [widgets.HTML("<b>Right Eye</b>"), right_eye_widget], layout=col_style,
)
eye_pair = widgets.VBox(
    [widgets.HBox([left_eye_col, right_eye_col]), pose_caption],
    layout=widgets.Layout(align_items="center"),
)
images_row = widgets.HBox(
    [maze_col, eye_pair],
    layout=widgets.Layout(margin="0 0 6px 0"),
)


def _form_label(text):
    h = widgets.HTML(f"<div>{text}</div>")
    h.add_class("form-field-label")
    return h


# Right column: just the picker (group → session_type → sessions list).
controls_col = widgets.VBox(
    [
        widgets.HTML('<div style="font-size:14px;"><b>Pick a session</b></div>'),
        _form_label("Group"),
        group_dd,
        _form_label("Session type"),
        session_type_dd,
        _form_label("Sessions"),
        sessions_select,
    ],
    layout=widgets.Layout(flex="1 1 0%", padding="10px", align_items="flex-start"),
)

# Left column: instructions + live info + trial progress, with all the
# action buttons (Run / Reset / Play / ← Back / Step →) and playback
# controls stacked beneath the Trial panel.
info_col = widgets.VBox(
    [
        widgets.HTML(
            f'<div style="font-size:12px; color:{MUTED_COLOR}; margin-bottom:8px;">'
            'Pick group → session type → session, click Run, then press Play.'
            '</div>'
        ),
        info_label,
        widgets.HTML('<div style="height:8px;"></div>'),
        trial_label,
        widgets.HTML('<div style="height:14px;"></div>'),
        widgets.HBox(
            [run_button, reset_button],
            layout=widgets.Layout(margin="0 0 6px 0", justify_content="flex-start"),
        ),
        widgets.HBox(
            [back_button, play_button, forward_button],
            layout=widgets.Layout(margin="0 0 6px 0", justify_content="flex-start"),
        ),
        _form_label("Progress"),
        progress,
        speed_slider,
        status_label,
    ],
    layout=widgets.Layout(flex="1 1 0%", padding="10px", align_items="flex-start"),
)

info_panel = widgets.HBox(
    [info_col, controls_col],
    layout=widgets.Layout(width="100%"),
)

divider = widgets.HTML(value="")
divider.add_class("section-divider")
ui = widgets.VBox([style_widget, images_row, divider, info_panel])
ui.add_class("maze-ui")

render_state()
display(ui)

## Try this

1. Pick a `PI+VC` rat on `Fixed Cue 1` and watch a clean acquisition session unfold — each correct well visit advances the trial counter and bumps the cumulative reward.
2. Try `PI` `Dark Train` to see what the env looks like with no visual cue (the cue tiles are gone).
3. Crank the speed slider to 120 to scan a session quickly; drop to 5 to study individual turns.

## What's next

- **03_compute_returns** — replay these same sessions to attach per-step `reward` and per-trial-with-ITI-start `return_to_go`, producing the input parquet for the Decision Transformer.
- **04_train_dt** — train DT on `actions_with_returns.parquet`.